In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier
import shap

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve
)

c:\Users\MICROSOFT SURFACE\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# LOAD DATA
# ============================================================

print("Loading datasets...")

df_healthy = pd.read_excel("dataset_Healthy (1).xlsx")
df_faulty = pd.read_excel("dataset_Faulty (1).xlsx")

print()

print("Healthy Dataset Shape :",df_healthy.shape)
print("Faulty Dataset Shape  :",df_faulty.shape)

print()

display(df_healthy.head())

Loading datasets...

Healthy Dataset Shape : (524289, 7)
Faulty Dataset Shape  : (524289, 7)



,I_Sw1_Healthy,I_Sw1_freqxaxis_Healthy,V_Sw1_Healthy,V_Sw1_freqxaxis_Healthy,StatorPh_CurrentAlpha_Healthy,StatorPh_CurrentBeta_Healthy,Conditions
0,-18.816096,0.000000,39.982615,0.000000,-3.835667,-34.396908,0
1,-5.504220,0.095367,35.385425,0.095367,-4.881387,-15.449395,0
2,-4.437106,0.190735,17.853357,0.190735,-3.863684,-29.877216,0
3,-5.026711,0.286102,24.997458,0.286102,-4.931154,-14.917556,0
4,-7.797710,0.381470,19.895151,0.381470,-4.117965,-40.415892,0


In [3]:
feature_count = df_healthy.shape[1]-1

headers=[]

for i in range(feature_count):

    headers.append(f"Feature_{i+1}")

headers.append("Condition")

df_healthy.columns=headers
df_faulty.columns=headers

print(df_healthy.columns)

Index(['Feature_1', 'Feature_2', 'Feature_3', 'Feature_4', 'Feature_5',
       'Feature_6', 'Condition'],
      dtype='str')


In [4]:
# ============================================================
# LEAK PROOF TRAIN TEST SPLIT
# ============================================================

train_h,test_h=train_test_split(

    df_healthy,

    test_size=0.20,

    shuffle=False

)

train_f,test_f=train_test_split(

    df_faulty,

    test_size=0.20,

    shuffle=False

)

print("Healthy Train :",train_h.shape)

print("Healthy Test  :",test_h.shape)

print()

print("Faulty Train :",train_f.shape)

print("Faulty Test  :",test_f.shape)

Healthy Train : (419431, 7)
Healthy Test  : (104858, 7)

Faulty Train : (419431, 7)
Faulty Test  : (104858, 7)


In [5]:
def extract_statistical_features(dataframe, window_size=20, step_size=5):
    X = dataframe.drop(columns=["Condition"]).values
    y = dataframe["Condition"].values

    X_engineered = []
    labels = []
    number_of_signals = X.shape[1]

    for start in range(0, len(dataframe) - window_size + 1, step_size):
        window = X[start:start + window_size]
        feature_vector = []
        for col in range(number_of_signals):
            signal = window[:, col]

            mean = np.mean(signal)
            std = np.std(signal)
            rms = np.sqrt(np.mean(signal**2))
            sig_min = np.min(signal)
            sig_max = np.max(signal)
            pkpk = sig_max - sig_min
            sig_skew = skew(signal)
            sig_kurt = kurtosis(signal)

            # Frequency-domain: dominant frequency + magnitude (excluding DC component)
            fft_vals = np.abs(np.fft.rfft(signal))
            fft_freqs = np.fft.rfftfreq(len(signal))
            dom_idx = np.argmax(fft_vals[1:]) + 1 if len(fft_vals) > 1 else 0
            dom_freq = fft_freqs[dom_idx]
            dom_mag = fft_vals[dom_idx]

            feature_vector.extend([
                mean, std, rms, sig_min, sig_max, pkpk,
                sig_skew, sig_kurt, dom_freq, dom_mag
            ])
        X_engineered.append(feature_vector)
        labels.append(y[start + window_size - 1])

    return np.array(X_engineered), np.array(labels)

In [6]:
def build_feature_names(number_of_signals):
    stat_names = ["mean", "std", "rms", "min", "max", "ptp", "skew", "kurt", "dom_freq", "dom_mag"]
    names = []
    for col in range(number_of_signals):
        for stat in stat_names:
            names.append(f"Feature_{col+1}_{stat}")
    return names

feature_names = build_feature_names(feature_count)
print(f"Total engineered features per window: {len(feature_names)}")

Total engineered features per window: 60


In [7]:
WINDOW_SIZE = 20
STEP_SIZE = 5

X_train_h, y_train_h = extract_statistical_features(train_h, WINDOW_SIZE, STEP_SIZE)
X_test_h, y_test_h = extract_statistical_features(test_h, WINDOW_SIZE, STEP_SIZE)

X_train_f, y_train_f = extract_statistical_features(train_f, WINDOW_SIZE, STEP_SIZE)
X_test_f, y_test_f = extract_statistical_features(test_f, WINDOW_SIZE, STEP_SIZE)

In [8]:
X_train = np.concatenate([X_train_h, X_train_f], axis=0)
y_train = np.concatenate([y_train_h, y_train_f], axis=0)

X_test = np.concatenate([X_test_h, X_test_f], axis=0)
y_test = np.concatenate([y_test_h, y_test_f], axis=0)

print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

Training Shape : (167766, 60)
Testing Shape  : (41936, 60)


In [9]:
print(df_healthy.iloc[:25, :6])

    Feature_1  Feature_2  Feature_3  Feature_4  Feature_5  Feature_6
0  -18.816096   0.000000  39.982615   0.000000  -3.835667 -34.396908
1   -5.504220   0.095367  35.385425   0.095367  -4.881387 -15.449395
2   -4.437106   0.190735  17.853357   0.190735  -3.863684 -29.877216
3   -5.026711   0.286102  24.997458   0.286102  -4.931154 -14.917556
4   -7.797710   0.381470  19.895151   0.381470  -4.117965 -40.415892
5  -35.940724   0.476837  22.206545   0.476837  -4.897452 -15.022380
6  -12.134014   0.572205  19.544832   0.572205  -4.554725 -37.488794
7  -17.175962   0.667572  20.189668   0.667572  -5.109605 -14.398261
8  -16.442343   0.762939  19.305098   0.762939  -5.203601 -34.058385
9  -29.520060   0.858307  18.789067   0.858307  -5.381284 -14.087603
10 -12.044918   0.953674  17.890460   0.953674  -5.919321 -26.851540
11 -13.739577   1.049042  16.713783   1.049042  -5.648351 -13.996910
12  -9.516972   1.144409  16.457578   1.144409  -6.523488 -27.238873
13  -8.974982   1.239777  14.54093

In [10]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

In [11]:
print("="*50)
print("Number of Engineered Features")
print("="*50)
print(X_train.shape[1])

Number of Engineered Features
60


In [12]:
import joblib
X_train_df.to_csv("X_train_features.csv", index=False)
X_test_df.to_csv("X_test_features.csv", index=False)

np.save("y_train.npy", y_train)
np.save("y_test.npy", y_test)

joblib.dump(scaler, "fault_scaler.joblib")
joblib.dump(feature_names, "feature_names.joblib")

print("Saved:")
print(" - X_train_features.csv, X_test_features.csv")
print(" - y_train.npy, y_test.npy")
print(" - fault_scaler.joblib")
print(" - feature_names.joblib")

Saved:
 - X_train_features.csv, X_test_features.csv
 - y_train.npy, y_test.npy
 - fault_scaler.joblib
 - feature_names.joblib
